01 — EDA Pendukung Preprocessing & Modeling (Pitri)
Peran: Data Preprocessing & Modeling
Kompetisi: Datavidia — Arkavidia

Pertanyaan utama:
1. Bagaimana bentuk dan tipe data pada shipments?
2. Kolom mana yang memiliki missing value, duplikasi, atau nilai tidak wajar?
3. Bagaimana distribusi target status_penanganan?
4. Fitur numerik dan kategorikal mana yang perlu diproses?
5. Apakah distribusi fitur train dan test cukup sebanding?
6. Apa keputusan awal preprocessing sebelum modeling?



#1. Import library dan menemukan lokasi dataset

In [22]:
%pip install seaborn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: C:\Users\Pitri Wulandari\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [23]:
import seaborn as sns

In [24]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 100)
sns.set_theme(style="whitegrid")

TARGET = "status_penanganan"

In [25]:
# Cari folder yang memiliki train/shipments.csv dan test/shipments.csv.
# Pencarian dimulai dari working directory dan folder induknya.
candidates = [Path.cwd(), *Path.cwd().parents]
DATA_ROOT = None

for base in candidates:
    for candidate in [base / "data" / "raw", base / "data"]:
        if (candidate / "train" / "shipments.csv").exists() and (candidate / "test" / "shipments.csv").exists():
            DATA_ROOT = candidate
            break
        # Dataset mungkin berada di dalam subfolder seperti data/raw/task-1-...
        if candidate.exists():
            matches = list(candidate.glob("**/train/shipments.csv"))
            for match in matches:
                possible_root = match.parent.parent
                if (possible_root / "test" / "shipments.csv").exists():
                    DATA_ROOT = possible_root
                    break
        if DATA_ROOT is not None:
            break
    if DATA_ROOT is not None:
        break

if DATA_ROOT is None:
    raise FileNotFoundError(
        "Folder dataset tidak ditemukan. Pastikan struktur memiliki "
        "data/raw/<folder-dataset>/train/shipments.csv dan test/shipments.csv."
    )

TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"
print("Lokasi dataset :", DATA_ROOT.resolve())
print("Train tersedia :", TRAIN_DIR.exists())
print("Test tersedia  :", TEST_DIR.exists())

Lokasi dataset : C:\Users\Pitri Wulandari\Documents\Datavidia\data\raw\task-1-datavidia-11-0\Task-1-Datavidia-11.0
Train tersedia : True
Test tersedia  : True


#2. Memuat tabel utama

In [26]:
train = pd.read_csv(TRAIN_DIR / "shipments.csv")
test = pd.read_csv(TEST_DIR / "shipments.csv")

print("Ukuran train:", train.shape)
print("Ukuran test :", test.shape)
display(train.head())

Ukuran train: (1243617, 38)
Ukuran test : (362245, 37)


,shipment_id,customer_id,courier_id,hub_id,package_id,promo_id,weight_kg,volume_l,declared_value,distance_km,ship_month,ship_dow,ship_hour,is_cod,is_promo,channel,pickup_type,goods_category,dest_region,insurance_opt,qty_items,coupon_count,gift_wrap,fragility_flag,order_to_ship_hours,prioritas_pengiriman,jumlah_catatan,flag_alamat_baru,berat_volumetrik,kota_asal,kota_tujuan,skor_prioritas_kirim,indeks_rute_sekunder,faktor_muatan_internal,varian_cuaca_rute,indeks_waktu_proses,skor_kualitas_cek,status_penanganan
0,SHP-AHSVFB3R,CUS-5VB7S4ZY,CRV-DZEF3GM9,HUB-MSVB47XA,PKG-NZ3YX33Z,TANPA_PROMO,3.75,7.9,16000,1277.6,3,6,16,0,0,aplikasi,agen,kesehatan,sumatera,1,2,1,0,0,1.4,2,0,0,6.69,banyuwangi,jayapura,39.50,43.72,37.79,47.06,57.00,55.10,normal
1,SHP-FUWWVBCQ,CUS-79DW3KY8,CRV-S8LG8JQP,HUB-B83PJTMD,PKG-GTVK7E6E,PRM-F82RASMM,1.46,3.4,23000,515.9,7,5,16,0,1,agen,dropoff,makanan,sulawesi,1,1,0,0,0,6.6,2,0,0,3.56,manado,palu,48.35,63.08,44.36,63.36,35.57,46.20,retur_pengirim
2,SHP-9EBN7MB9,CUS-642HAPWJ,CRV-EQ3SX46G,HUB-AGDQJQVF,PKG-QPNPLAVN,TANPA_PROMO,3.30,8.0,119000,653.1,6,1,12,0,0,agen,agen,kesehatan,sulawesi,1,3,0,0,0,8.4,1,1,0,7.18,bandung,solo,59.01,55.77,55.04,54.16,48.49,36.90,normal
3,SHP-5LK9F3MX,CUS-QDZSXZWD,CRV-XYHRH2DD,HUB-22X96KLN,PKG-2Y2RN6AW,TANPA_PROMO,4.77,7.5,98000,434.5,2,5,17,0,0,agen,dropoff,kesehatan,jawa_barat,0,1,0,0,1,6.6,2,1,0,12.34,surabaya,pekanbaru,47.49,49.37,61.23,42.10,50.49,60.30,retur_pengirim
4,SHP-6XV4H3CP,CUS-QADQDZQ3,CRV-LQWU6RFS,HUB-YZUUKVTW,PKG-R3P3KWCQ,PRM-EAXBNEFE,3.24,6.4,86000,997.4,8,1,10,0,1,aplikasi,jemput,fashion,jawa_timur,1,1,1,0,1,3.4,1,0,0,3.98,kudus,purwokerto,47.34,56.25,54.06,52.92,44.46,45.72,konfirmasi_alamat


#3. Struktur kolom dan tipe data

In [27]:
schema = pd.DataFrame({
    "tipe_data": train.dtypes.astype(str),
    "jumlah_nilai_unik": train.nunique(dropna=True),
    "missing": train.isna().sum(),
    "missing_%": (train.isna().mean() * 100).round(2)
}).sort_values(["missing_%", "jumlah_nilai_unik"], ascending=[False, False])

display(schema)
print("\nJumlah kolom train:", train.shape[1])
print("Kolom target tersedia:", TARGET in train.columns)

,tipe_data,jumlah_nilai_unik,missing,missing_%
volume_l,float64,1393,37012,2.98
shipment_id,str,1243617,0,0.00
customer_id,str,508086,0,0.00
courier_id,str,23806,0,0.00
distance_km,float64,18525,0,0.00
faktor_muatan_internal,float64,8708,0,0.00
varian_cuaca_rute,float64,8689,0,0.00
indeks_rute_sekunder,float64,8686,0,0.00
indeks_waktu_proses,float64,8665,0,0.00
skor_prioritas_kirim,float64,8661,0,0.00



Jumlah kolom train: 38
Kolom target tersedia: True


#4.Pemeriksaan duplikasi dan ID

In [29]:
# Pemeriksaan duplikasi menggunakan kolom ID agar hemat memori

print("PEMERIKSAAN DUPLIKASI DATA")
print("=" * 40)

if "shipment_id" in train.columns:
    print("ID duplikat train:",
          train["shipment_id"].duplicated().sum())

if "shipment_id" in test.columns:
    print("ID duplikat test:",
          test["shipment_id"].duplicated().sum())

# Pemeriksaan duplikasi berdasarkan sampel data
sample_train = train.sample(
    n=min(10000, len(train)),
    random_state=42
)

print("Duplikasi pada sampel 10.000 baris:",
      sample_train.duplicated().sum())

del sample_train

PEMERIKSAAN DUPLIKASI DATA
ID duplikat train: 0
ID duplikat test: 0
Duplikasi pada sampel 10.000 baris: 0


#5. Analisis missing value

In [ ]:
missing = (train.isna().sum().rename("jumlah_missing").to_frame())
missing["persentase"] = (missing["jumlah_missing"] / len(train) * 100).round(2)
missing = missing[missing["jumlah_missing"] > 0].sort_values("persentase", ascending=False)

if missing.empty:
    print("Tidak ditemukan missing value pada tabel shipments train.")
else:
    display(missing)
    plt.figure(figsize=(10, max(4, len(missing) * 0.35)))
    sns.barplot(data=missing.reset_index(), x="persentase", y="index")
    plt.title("Persentase Missing Value — Shipments Train")
    plt.xlabel("Missing (%)")
    plt.ylabel("Kolom")
    plt.tight_layout()
    plt.show()

#6. Statistik deskriptif fitur numerik

In [ ]:
numeric_cols = train.select_dtypes(include=np.number).columns.tolist()
numeric_features = [c for c in numeric_cols if c != TARGET]

if numeric_features:
    display(train[numeric_features].describe().T)
else:
    print("Tidak ada fitur numerik yang terdeteksi.")

#7. Distribusi target dan ketidakseimbangan kelas

In [ ]:
if TARGET not in train.columns:
    print(f"Kolom target '{TARGET}' tidak ditemukan. Periksa nama target pada dataset.")
else:
    target_dist = train[TARGET].value_counts(dropna=False).rename_axis("kelas").reset_index(name="jumlah")
    target_dist["persentase"] = (target_dist["jumlah"] / len(train) * 100).round(2)
    display(target_dist)

    plt.figure(figsize=(10, 5))
    ax = sns.countplot(data=train, x=TARGET, order=train[TARGET].value_counts().index)
    ax.set_title("Distribusi Target status_penanganan")
    ax.set_xlabel("Kelas target")
    ax.set_ylabel("Jumlah data")
    plt.xticks(rotation=30, ha="right")
    for container in ax.containers:
        ax.bar_label(container, fmt="%.0f")
    plt.tight_layout()
    plt.show()

    counts = train[TARGET].value_counts()
    if len(counts) > 1:
        print("Rasio kelas terbesar : terkecil =", round(counts.max() / counts.min(), 2))

#8. Distribusi fitur numerik dan indikasi outlier

In [ ]:
if numeric_features:
    ncols = 2
    nrows = int(np.ceil(len(numeric_features) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, numeric_features):
        sns.histplot(train[col].dropna(), kde=True, ax=ax)
        ax.set_title(f"Distribusi {col}")
    for ax in axes[len(numeric_features):]:
        ax.remove()
    plt.tight_layout()
    plt.show()

    outlier_rows = []
    for col in numeric_features:
        q1, q3 = train[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        n_out = ((train[col] < lower) | (train[col] > upper)).sum()
        outlier_rows.append([col, lower, upper, int(n_out), round(n_out / len(train) * 100, 2)])
    outlier_summary = pd.DataFrame(
        outlier_rows, columns=["fitur", "batas_bawah_IQR", "batas_atas_IQR", "jumlah_outlier", "persentase_%"]
    ).sort_values("persentase_%", ascending=False)
    display(outlier_summary)

#9. Distribusi fitur kategorikal

In [ ]:
categorical_cols = train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
categorical_features = [c for c in categorical_cols if c != TARGET]

cat_summary = []
for col in categorical_features:
    cat_summary.append({
        "fitur": col,
        "jumlah_kategori": train[col].nunique(dropna=True),
        "missing": train[col].isna().sum(),
        "kategori_teratas": train[col].value_counts(dropna=False).head(3).to_dict()
    })
display(pd.DataFrame(cat_summary))

# Visualisasi beberapa fitur dengan kategori terbanyak yang masih mudah dibaca.
for col in categorical_features[:6]:
    top_values = train[col].value_counts().head(12).index
    plt.figure(figsize=(9, 4))
    sns.countplot(data=train[train[col].isin(top_values)], y=col, order=top_values)
    plt.title(f"12 kategori terbanyak — {col}")
    plt.tight_layout()
    plt.show()

#10. Korelasi fitur numerik

In [ ]:
if len(numeric_features) >= 2:
    corr = train[numeric_features].corr(numeric_only=True)
    plt.figure(figsize=(12, 9))
    sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
    plt.title("Korelasi Antar Fitur Numerik")
    plt.tight_layout()
    plt.show()
    display(corr.round(2))
else:
    print("Fitur numerik kurang dari dua; korelasi tidak dapat dihitung.")

#11. Perbandingan distribusi train dan test

In [ ]:
common_cols = [c for c in train.columns if c in test.columns and c != TARGET]
comparison = []

for col in common_cols:
    if pd.api.types.is_numeric_dtype(train[col]) and pd.api.types.is_numeric_dtype(test[col]):
        comparison.append({
            "fitur": col, "tipe": "numerik",
            "train_mean": train[col].mean(), "test_mean": test[col].mean(),
            "train_median": train[col].median(), "test_median": test[col].median(),
            "train_missing_%": train[col].isna().mean() * 100,
            "test_missing_%": test[col].isna().mean() * 100
        })
    else:
        comparison.append({
            "fitur": col, "tipe": "kategorikal",
            "train_mean": np.nan, "test_mean": np.nan,
            "train_median": np.nan, "test_median": np.nan,
            "train_missing_%": train[col].isna().mean() * 100,
            "test_missing_%": test[col].isna().mean() * 100
        })

display(pd.DataFrame(comparison))

#12. Catatan awal untuk preprocessing dan modeling

In [ ]:
print("CHECKLIST PREPROCESSING")
print("-" * 55)

missing_cols = train.columns[train.isna().any()].tolist()
print("1. Kolom dengan missing value:", missing_cols if missing_cols else "Tidak ada")

high_cardinality = [
    c for c in categorical_features if train[c].nunique(dropna=True) > 50
]
print("2. Kategorikal dengan >50 kategori:", high_cardinality if high_cardinality else "Tidak ada")

constant_cols = [c for c in train.columns if train[c].nunique(dropna=False) <= 1]
print("3. Kolom konstan:", constant_cols if constant_cols else "Tidak ada")

if TARGET in train.columns:
    print("4. Target:", TARGET)
    print("5. Jumlah kelas target:", train[TARGET].nunique(dropna=True))

print("\nCatatan:")
print("- Jangan menghapus outlier secara otomatis; validasi dahulu apakah nilainya tidak masuk akal.")
print("- Imputasi, encoding, scaling, dan seleksi fitur harus dipelajari dari data train.")
print("- Terapkan transformasi yang sama ke validation dan test.")
print("- Hindari fitur yang membocorkan target atau informasi setelah hasil pengiriman diketahui.")
print("- Untuk tabel one-to-many, agregasikan dahulu per shipment_id sebelum join ke shipments.")